# 15장. 라즈베리파이 배포

In [1]:
# [준비 셀] 모든 노트북의 첫 코드 셀입니다. 항상 먼저 실행하세요.
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[1]          # docs/manual → 프로젝트 루트
assert (ROOT / "src/bearing_kd").exists(), "docs/manual 폴더에서 노트북을 열어야 합니다"
CODE = ROOT / "docs/manual/code"                 # 챕터별 완성 스크립트 위치
sys.path.insert(0, str(CODE))                    # 스크립트의 함수를 import 해서 쓰기 위해
print("프로젝트 루트 확인 완료:", ROOT.name)

프로젝트 루트 확인 완료: vib-onnx


이제 ONNX 모델을 **Raspberry Pi 5**에서 돌립니다. 이 장의 셀은 Mac에서 **배포 묶음**을 만들고, 라즈베리파이에서 할 일은 터미널 명령으로 안내합니다.

```text
[Mac]  ONNX(14장 정리본) + 검증 데이터 + 라즈베리파이용 스크립트  ──scp──▶  [Raspberry Pi 5]
                                                                            python pi_infer.py verify
```

## 15.1 라즈베리파이에서의 추론 순서

```text
진동 원본 (64 kHz)
  → 4장과 똑같은 전처리 → 1초 윈도우 [4000]
  → 청크 8개 [8,1,512] + mask
  → ONNX 8번 실행 (청크 1개씩)  → logits 8개
  → 평균 → argmax → 정상/고장
```

**전처리 계약을 지켜야 합니다.** 필터 차수, 샘플링 레이트, 윈도우 길이 중 하나라도 다르면 모델은 학습 때와 다른 데이터를 보게 됩니다.

## 15.2 배포 묶음 만들기 (Mac)

In [2]:
%run code/ch15_pi_deploy.py

배포 묶음: outputs/manual/pi_bundle
  pi_infer.py                 0.0 MB
  pi_verify_data.npz          0.9 MB
  student.onnx                0.5 MB
  student.onnx.data         602.2 MB

검증용 윈도우 57개, 이 컴퓨터에서 윈도우당 25.7 ms
  이 컴퓨터                  청크   3.2 ms → 윈도우   25.7 ms → 0.5초 예산의 5% 사용 (100% 미만이면 실시간 가능)
  Raspberry Pi 5 (실측)    청크  38.9 ms → 윈도우  311.2 ms → 0.5초 예산의 62% 사용 (100% 미만이면 실시간 가능)


| 파일 | 역할 |
|---|---|
| `student.onnx` + `student.onnx.data` | 모델. **같은 폴더, 이름 그대로** |
| `pi_verify_data.npz` | 전처리된 윈도우 57개 + Mac에서 계산한 기준 logits (라즈베리파이 결과와 비교용) |
| `pi_infer.py` | 라즈베리파이용 스크립트. `numpy`, `onnxruntime`만 사용 (원본 `.mat` 처리 시 `scipy`) |

`.npz`는 NumPy 배열 여러 개를 파일 하나에 묶은 형식입니다.

In [3]:
bundle = ROOT / "outputs/manual/pi_bundle"
data = np.load(bundle / "pi_verify_data.npz")
print({k: data[k].shape for k in data.files})
print("첫 윈도우 출처:", str(data["names"][0]), " 정답", int(data["labels"][0]), " 기준 logits", data["expected_logits"][0].round(3))

{'windows': (57, 4000), 'labels': (57,), 'expected_logits': (57, 2), 'names': (57,)}
첫 윈도우 출처: K002/N09_M07_F10_K002_10_windows.npy:0  정답 0  기준 logits [ 1.965 -1.863]


라즈베리파이로 옮기기 전에 **Mac에서 먼저** 라즈베리파이 스크립트를 돌려 봅니다.

In [4]:
import subprocess, sys
print(subprocess.run([sys.executable, "pi_infer.py", "verify"], cwd=bundle, capture_output=True, text=True).stdout)

윈도우 57개 | Mac 대비 최대 차이 0.0e+00 | 예측 일치 57/57
윈도우당 24 ms (실시간 한계 500 ms) → 실시간 가능
결과: PASS



## 15.3 라즈베리파이 준비 (라즈베리파이 터미널에서)

라즈베리파이 OS(64-bit)에는 Python이 이미 있습니다. 검증한 환경은 **Python 3.13**입니다. 학습은 하지 않으므로 PyTorch와 MOMENT는 필요 없습니다.

```bash
mkdir -p ~/vib-onnx && cd ~/vib-onnx
python3 -m venv .venv
source .venv/bin/activate
python -m pip install numpy onnxruntime scipy
```

> 새 터미널을 열 때마다 `source .venv/bin/activate`를 해야 합니다. 안 하면 `No module named 'onnxruntime'` 에러가 납니다. 또는 `~/vib-onnx/.venv/bin/python`으로 직접 실행하세요.

## 15.4 파일 옮기기 (Mac 터미널에서)

`<user>`와 `<pi-address>`는 본인 라즈베리파이의 사용자 이름과 IP 주소로 바꾸세요.
```bash
scp -r outputs/manual/pi_bundle <user>@<pi-address>:~/vib-onnx/
```

## 15.5 검증 실행 (라즈베리파이에서)

```bash
cd ~/vib-onnx/pi_bundle
~/vib-onnx/.venv/bin/python pi_infer.py verify
```

이 프로젝트의 실제 Raspberry Pi 5 결과 (v3 모델, 윈도우 56개):
```text
Mac 대비 logits 최대 차이: 7.153e-07  (기준 1e-4 이하면 통과)
Mac과 예측 일치: 56/56
청크당 평균 38.93 ms, 윈도우당 311.4 ms
결과: PASS
```

- **차이 7e-7**: CPU 종류(Apple Silicon ↔ ARM Cortex-A76)에 따른 부동소수점 반올림 차이입니다. 판정에는 영향이 없습니다.
- 원본 `.mat` 파일로 판정하려면: `python pi_infer.py mat 파일경로.mat`

## 15.6 실시간으로 돌 수 있나?

윈도우는 0.5초(2,000샘플)마다 하나씩 새로 생깁니다. 그러니 **윈도우 하나를 0.5초 안에** 처리해야 밀리지 않습니다.

In [5]:
from ch15_pi_deploy import realtime_budget
for name, ms in [("Mac (측정)", 3.2), ("Raspberry Pi 5 (측정)", 38.9), ("가상의 느린 장치", 70.0)]:
    per_window, load = realtime_budget(ms)
    print(f"{name:22s} 청크 {ms:5.1f} ms × 8 = 윈도우 {per_window:6.1f} ms → 0.5초 예산의 {load:4.0%} {'✅' if load < 1 else '❌ 밀림'}")

Mac (측정)               청크   3.2 ms × 8 = 윈도우   25.6 ms → 0.5초 예산의   5% ✅
Raspberry Pi 5 (측정)    청크  38.9 ms × 8 = 윈도우  311.2 ms → 0.5초 예산의  62% ✅
가상의 느린 장치              청크  70.0 ms × 8 = 윈도우  560.0 ms → 0.5초 예산의 112% ❌ 밀림


속도에 영향을 주는 것:
- **스레드 수** (`intra_op_num_threads`): 라즈베리파이 5는 4코어입니다. 2와 4를 비교해 보세요.
- **발열**: 오래 돌리면 CPU가 뜨거워져 속도가 떨어질 수 있습니다(방열판, 팬 권장).
- **모델 크기**: 4 kHz, 8청크로 설계한 이유가 이 속도 예산 때문입니다(4장).

## 15.7 아직 남은 일: 실시간 스트리밍

지금 전처리의 `sosfiltfilt`는 **녹음이 끝난 뒤** 앞뒤로 필터를 거는 방식입니다. 센서에서 실시간으로 들어오는 신호에는 쓸 수 없습니다. 실시간으로 쓰려면:
- 한쪽 방향 필터(`sosfilt`)에 상태(`zi`)를 이어 가며 적용해야 합니다.
- 그러면 학습 때(양방향)와 전처리가 달라지므로, **같은 방식으로 다시 학습**하거나 차이를 검증해야 합니다.
- 센서의 실제 샘플링 레이트도 확인해야 합니다. 4 kHz 미만이면 이 모델의 전처리 계약을 지킬 수 없습니다.

## 15장 완성 코드: `code/ch15_pi_deploy.py`

In [6]:
from IPython.display import Code
Code(filename=str(CODE / "ch15_pi_deploy.py"), language="python")

"""15장 완성 코드: 라즈베리파이로 가져갈 배포 묶음을 만들고, 실시간 처리 가능 여부를 계산한다.

Mac(학습 컴퓨터)에서 실행:
    python docs/manual/code/ch15_pi_deploy.py
→ outputs/manual/pi_bundle/ 에 다음을 만든다
    student.onnx, student.onnx.data   모델 (이름 바꾸지 말 것)
    pi_verify_data.npz                 전처리된 윈도우 + Mac에서 계산한 기준 logits
    pi_infer.py                        라즈베리파이용 추론·검증 스크립트 (numpy + onnxruntime만 사용)

라즈베리파이에서 실행:
    python pi_infer.py verify                  # Mac 결과와 같은지 + 속도
    python pi_infer.py mat 파일.mat            # 원본 .mat을 전처리해서 판정 (scipy 필요)
"""

import argparse
import shutil
import time
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[3]

PI_SCRIPT = r'''"""Raspberry Pi: ONNX 진동 판정 + 검증. 필요 패키지: numpy, onnxruntime (mat 모드는 scipy도)."""
import sys
import time
from pathlib import Path

import numpy as np
import onnxruntime as ort

HERE = Path(__file__).resolve().parent
WINDOW, CHUNK = 4000, 512


def session(threads=4):
    opts = ort.SessionOptions()
    opts.intra_op_num_threads = threads
    return ort.InferenceSession(str(HERE / "student.onnx"), sess_options=opts, providers=["CPUExecutionProvider"])


def predict_window(sess, window):
    chunks = np.pad(window.astype(np.float32), (0, 96)).reshape(8, 1, CHUNK)
    masks = (np.arange(8 * CHUNK) < WINDOW).astype(np.float32).reshape(8, CHUNK)
    logits = [sess.run(["logits"], {"x_enc": chunks[i:i + 1], "input_mask": masks[i:i + 1]})[0][0] for i in range(8)]
    return np.mean(logits, axis=0)                       # 청크 8개 logits 평균 = 윈도우 판정


def preprocess_mat(path):
    """학습과 똑같은 전처리 (4장): 1 kHz 저역통과(8차, 양방향) → 4 kHz → 1초 윈도우, 0.5초 이동."""
    from scipy.io import loadmat
    from scipy.signal import butter, resample_poly, sosfiltfilt
    path = Path(path)
    record = loadmat(path, squeeze_me=True, struct_as_record=False)[path.stem]
    x = next(np.asarray(i.Data, dtype=np.float32).ravel() for i in np.atleast_1d(record.Y) if str(i.Name) == "vibration_1")
    sos = butter(8, 1000 / 32000, btype="lowpass", output="sos")
    x = resample_poly(sosfiltfilt(sos, x).astype(np.float32), 1, 16).astype(np.float32)
    return np.stack([x[s:s + WINDOW] for s in range(0, len(x) - WINDOW + 1, 2000)])


def verify(sess):
    data = np.load(HERE / "pi_verify_data.npz")
    start = time.perf_counter()
    actual = np.array([predict_window(sess, w) for w in data["windows"]])
    per_window = (time.perf_counter() - start) / len(actual) * 1000
    diff = float(np.abs(actual - data["expected_logits"]).max())
    same = int((actual.argmax(1) == data["expected_logits"].argmax(1)).sum())
    print(f"윈도우 {len(actual)}개 | Mac 대비 최대 차이 {diff:.1e} | 예측 일치 {same}/{len(actual)}")
    print(f"윈도우당 {per_window:.0f} ms (실시간 한계 500 ms) → {'실시간 가능' if per_window < 500 else '실시간 불가'}")
    print("결과:", "PASS" if diff <= 1e-4 and same == len(actual) else "FAIL")


if __name__ == "__main__":
    sess = session()
    if len(sys.argv) >= 3 and sys.argv[1] == "mat":
        for i, w in enumerate(preprocess_mat(sys.argv[2])):
            logit = predict_window(sess, w)
            print(f"윈도우 {i}: logits [{logit[0]:+.3f}, {logit[1]:+.3f}] → {'고장' if logit.argmax() else '정상'}")
    else:
        verify(sess)
'''


def predict_window(sess, window):
    """PI_SCRIPT의 predict_window와 같은 계산: 청크 8개 → logits 평균."""
    chunks = np.pad(window.astype(np.float32), (0, 96)).reshape(8, 1, 512)
    masks = (np.arange(8 * 512) < 4000).astype(np.float32).reshape(8, 512)
    return np.mean([sess.run(["logits"], {"x_enc": chunks[i:i + 1], "input_mask": masks[i:i + 1]})[0][0]
                    for i in range(8)], axis=0)


def make_bundle(onnx_path, windows_files, out_dir):
    from ch06_dataset import WindowDataset
    import onnxruntime as ort

    out_dir.mkdir(parents=True, exist_ok=True)
    for f in (onnx_path, Path(str(onnx_path) + ".data")):
        shutil.copy2(f, out_dir / f.name)                 # .onnx와 .onnx.data는 같은 폴더, 같은 이름
    (out_dir / "pi_infer.py").write_text(PI_SCRIPT)

    ds = WindowDataset(win